<a href="https://colab.research.google.com/github/eunyeongkimm/multimodal_user_needs_understanding/blob/main/qwen2_5_omni_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== 셀 1: 환경 설치 =====
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q qwen-omni-utils librosa soundfile
!pip install -U -q autoawq
!nvidia-smi --query-gpu=name,memory.total --format=csv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
!pip install -U -q "compressed-tensors>=0.15.0"

In [ ]:
# ===== 셀 2: Drive mount + 경로/데이터 로드 =====
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, os, glob

AUDIO_ROOT = '/content/drive/MyDrive/audio_seg'
PARQUET_DIR = '/content/drive/MyDrive/audio_seg_2'

# 파일명 확인용 — 실제 parquet 이름이 이거 맞는지 출력 보고
print(os.listdir(PARQUET_DIR))

manifest = pd.read_parquet(os.path.join(PARQUET_DIR, 'audio_seg_manifest.parquet'))
gold     = pd.read_parquet(os.path.join(PARQUET_DIR, 'gold_actual_batch1_final.parquet'))

print("manifest:", manifest.shape, "| gold:", gold.shape)
print(manifest.columns.tolist())
print(gold.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['gold_actual_batch1_final.parquet', 'audio_seg_manifest.parquet']
manifest: (1119, 11) | gold: (19847, 5)
['call_id', 'utt_idx', 'wav_path', 'dialog_idx', 'duration', 'text', 'n_customer_utt', 'wav_duration', 'sample_rate', 'n_channels', 'error']
['call_id', 'label', 'prompt_version', 'model', 'source']


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated()/1e9, "GB used /", torch.cuda.get_device_properties(0).total_memory/1e9, "GB total")

0.0 GB used / 42.405855232 GB total


In [ ]:
# ===== 셀 3: Qwen2.5-Omni-7B 로드 (4bit, talker 유지) =====
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map="cuda")

model.eval()
print("loaded, GPU used:", round(torch.cuda.memory_allocated()/1e9, 1), "GB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2447 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-7B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded, GPU used: 12.4 GB


In [ ]:
# ===== 셀 4: 추론 함수 + 5콜 테스트 (요약 진단판) =====
import soundfile as sf, os, re, time

CATEGORIES = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]

SYSTEM = "당신은 콜센터 통화 분류기입니다. 고객의 초반 발화 음성을 듣고 의도를 분류합니다."

INSTRUCTION = f"""아래는 한 고객의 통화 초반 발화들(음성)입니다. 순서대로 듣고,
고객의 의도를 다음 7개 중 정확히 하나로 분류하세요:
{", ".join(CATEGORIES)}

먼저 음성에서 들리는 내용을 한 문장으로 요약한 뒤, 마지막에 정답을 쓰세요.
아래 형식을 지키세요:
요약: (고객이 실제로 뭐라고 말하는지)
정답: (위 7개 중 하나)"""

def get_call_wavs(call_id):
    rows = manifest[manifest.call_id == call_id].sort_values("utt_idx")
    paths = []
    for _, r in rows.iterrows():
        p = os.path.join(AUDIO_ROOT, call_id, os.path.basename(r.wav_path))
        if os.path.exists(p):
            paths.append(p)
    return paths

def parse_pred(resp):
    m = re.search(r"정답\s*[:：]\s*([가-힣]+)", resp)
    if m and m.group(1).strip() in CATEGORIES:
        return m.group(1).strip()
    m = re.search(r"<카테고리>\s*(.+?)\s*</카테고리>", resp)
    if m and m.group(1).strip() in CATEGORIES:
        return m.group(1).strip()
    for c in CATEGORIES:
        if f"<{c}>" in resp:
            return c
    tail = resp[-100:]
    for c in CATEGORIES:
        if c in tail:
            return c
    for c in CATEGORIES:
        if c in resp:
            return c
    return None

def predict_call(call_id):
    wavs = get_call_wavs(call_id)
    if not wavs:
        return None, "no_audio"
    content = []
    for i, w in enumerate(wavs, 1):
        content.append({"type": "text", "text": f"발화{i}:"})
        content.append({"type": "audio", "audio": w})
    content.append({"type": "text", "text": INSTRUCTION})
    conv = [
        {"role": "system", "content": [{"type":"text","text":SYSTEM}]},
        {"role": "user", "content": content},
    ]
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    audios = [sf.read(w)[0] for w in wavs]
    inputs = processor(text=text, audio=audios, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=2048, do_sample=False, return_audio=False)
    if isinstance(out, (tuple, list)):
        out = out[0]
    resp = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return parse_pred(resp), resp

# ---- 5콜 테스트 (요약 확인) ----
gold_map = dict(zip(gold.call_id, gold.label))
for cid in manifest.call_id.unique()[:5]:
    t0 = time.time()
    pred, raw = predict_call(cid)
    print(f"\n=== {cid}  gold={gold_map.get(cid)}  pred={pred}  ({time.time()-t0:.1f}s) ===")
    print(raw[-400:])


=== J16_S000434  gold=서비스이용  pred=불만제기  (2.0s) ===
요약: 고객이 제품에 대한 불만을 제기하고 있습니다.

정답: 불만제기

=== J16_S000633  gold=서비스이용  pred=환불요청  (2.6s) ===
요약: 고객이 기기 장정에 대해 이야기하고, 구분이 막혀서 불편함을 표현합니다.

정답: 환불요청

=== J16_S000687  gold=서비스이용  pred=배송확인  (1.8s) ===
요약: 고객이 배송에 대한 확인을 요청합니다.

정답: 배송확인

=== J16_S000727  gold=서비스이용  pred=주문취소  (1.4s) ===
요약: 고객이 주문을 취소하려고 합니다.
정답: 주문취소

=== J16_S000746  gold=서비스이용  pred=불만제기  (2.6s) ===
요약: 고객이 상품에 대한 불만을 제기하고, 상담원이 해결책을 제안하려고 합니다.

정답: 불만제기


In [ ]:
# ===== 셀 5: 250 전체 추론 + 저장 =====
import pandas as pd, time

call_ids = list(manifest.call_id.unique())
rows = []
t_start = time.time()
for i, cid in enumerate(call_ids):
    pred, raw = predict_call(cid)
    rows.append({"call_id": cid, "pred_qwen": pred, "gold": gold_map.get(cid), "raw_tail": raw[-200:]})
    if (i+1) % 25 == 0:
        print(f"{i+1}/250  ({time.time()-t_start:.0f}s)")

df = pd.DataFrame(rows)
save_path = "/content/drive/MyDrive/audio_seg_2/qwen25_7b_predictions.parquet"
df.to_parquet(save_path)
print("saved:", save_path)

# 즉석 성능
parsed = df.dropna(subset=["pred_qwen"])
acc = (parsed.pred_qwen == parsed.gold).mean()
print(f"\n파싱 성공: {len(parsed)}/250, 파싱실패: {df.pred_qwen.isna().sum()}")
print(f"accuracy: {acc:.3f}")
print("\n예측 분포:\n", df.pred_qwen.value_counts())
print("\ngold 분포:\n", df.gold.value_counts())

25/250  (131s)
50/250  (250s)
75/250  (390s)
100/250  (527s)
125/250  (662s)
150/250  (793s)
175/250  (922s)
200/250  (1053s)
225/250  (1190s)
250/250  (1335s)
saved: /content/drive/MyDrive/audio_seg_2/qwen25_7b_predictions.parquet

파싱 성공: 249/250, 파싱실패: 1
accuracy: 0.173

예측 분포:
 pred_qwen
배송확인     94
주문취소     71
환불요청     38
불만제기     24
교환반품     11
구매진행      6
서비스이용     5
Name: count, dtype: int64

gold 분포:
 gold
환불요청     87
서비스이용    51
불만제기     50
배송확인     24
교환반품     19
구매진행     11
주문취소      8
Name: count, dtype: int64
